## Building A Chatbot
In this video We'll go over an example of how to design and implement an LLM-powered chatbot. This chatbot will be able to have a conversation and remember previous interactions.

Note that this chatbot that we build will only use the language model to have a conversation. There are several other related concepts that you may be looking for:

- Conversational RAG: Enable a chatbot experience over an external source of data
- Agents: Build a chatbot that can take actions

This video tutorial will cover the basics which will be helpful for those two more advanced topics.

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key=os.getenv("GROQ_API_KEY")

In [2]:
# from langchain_ollama import ChatOllama
from langchain_groq import ChatGroq
# model=ChatOllama(model="gemma:2b")
model=ChatGroq(api_key=groq_api_key,model="groq/compound")
# model
response =model.invoke("Hello, how are you?")
print(response.content)

d:\udemy\python\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Hello! I'm doing well, thank you. How can I help you today?


In [3]:
from langchain_core.messages import HumanMessage
model.invoke([HumanMessage(content="Hi , My name is Akshat and I am a UG Student")])

AIMessage(content='Hello Akshat! Nice to meet you. How can I help you today?', additional_kwargs={'reasoning_content': '<Think>\n\n</Think>'}, response_metadata={'token_usage': {'completion_tokens': 79, 'prompt_tokens': 258, 'total_tokens': 337, 'completion_time': 0.167062, 'completion_tokens_details': None, 'prompt_time': 0.009963, 'prompt_tokens_details': None, 'queue_time': 0.092295, 'total_time': 0.177025}, 'model_name': 'groq/compound', 'system_fingerprint': None, 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ea5ad-ec0b-7f03-bfd2-ebb4ef323e2a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 258, 'output_tokens': 79, 'total_tokens': 337})

In [4]:
from langchain_core.messages import AIMessage
model.invoke(
    [
        HumanMessage(content="Hi , My name is Akshat and I am a UG Student"),
        AIMessage(content="Hello Akshat! It's a pleasure to meet you. What can I do for you today?\n"),
        HumanMessage(content="Hey What's my name and what do I do?")
    ]
)

AIMessage(content='Your name is **Akshat**, and you mentioned that you’re a **undergraduate (UG) student**. Let me know if there’s anything specific you’d like help with!', additional_kwargs={'reasoning_content': '<Think>\n\n</Think>'}, response_metadata={'token_usage': {'completion_tokens': 128, 'prompt_tokens': 338, 'total_tokens': 466, 'completion_time': 0.276462, 'completion_tokens_details': None, 'prompt_time': 0.016408, 'prompt_tokens_details': None, 'queue_time': 0.092786, 'total_time': 0.29287}, 'model_name': 'groq/compound', 'system_fingerprint': None, 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ea5ad-ed49-70b0-bdc2-876928b84049-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 338, 'output_tokens': 128, 'total_tokens': 466})

### Message History
We can use a Message History class to wrap our model and make it stateful. This will keep track of inputs and outputs of the model, and store them in some datastore. Future interactions will then load those messages and pass them into the chain as part of the input. Let's see how to use this!

In [5]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store={}

def get_session_history(session_id:str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id]=ChatMessageHistory()
    return store[session_id]

with_message_history=RunnableWithMessageHistory(model,get_session_history)

d:\udemy\python\venv\Lib\site-packages\IPython\core\interactiveshell.py:3747: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [6]:
config={"configurable":{"session_id":"chat1"}}

In [7]:
response=with_message_history.invoke(
    [HumanMessage(content="Hi , My name is Akshat and I am a UG Student")],
    config=config
)

In [8]:
response.content

'Hello Akshat! Nice to meet you. How can I help you today?'

In [10]:
with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config,
)

AIMessage(content='Your name is Akshat.', additional_kwargs={'reasoning_content': '<Think>\n\n</Think>'}, response_metadata={'token_usage': {'completion_tokens': 60, 'prompt_tokens': 320, 'total_tokens': 380, 'completion_time': 0.129048, 'completion_tokens_details': None, 'prompt_time': 0.01566, 'prompt_tokens_details': None, 'queue_time': 0.091903, 'total_time': 0.144708}, 'model_name': 'groq/compound', 'system_fingerprint': None, 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ea5ae-5756-7353-9da2-814c29d24fe2-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 320, 'output_tokens': 60, 'total_tokens': 380})

In [11]:
## change the config-->session id
config1={"configurable":{"session_id":"chat2"}}
response=with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config1
)
response.content

'I’m sorry, but you haven’t shared your name with me, so I don’t have that information. If you’d like me to address you by a particular name, just let me know!'

In [12]:
response=with_message_history.invoke(
    [HumanMessage(content="Hey My name is John")],
    config=config1
)
response.content

"Nice to meet you, John! Let me know if there's anything I can help you with."

In [13]:
response=with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config1
)
response.content

'Your name is John.'

### Prompt templates
Prompt Templates help to turn raw user information into a format that the LLM can work with. In this case, the raw user input is just a message, which we are passing to the LLM. Let's now make that a bit more complicated. First, let's add in a system message with some custom instructions (but still taking messages as input). Next, we'll add in more input besides just the messages.

In [15]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
prompt=ChatPromptTemplate.from_messages(
    [
        ("system","You are a helpful assistant. Answer all the question to the best of your ability"),
        MessagesPlaceholder(variable_name="messages")
    ]
)

chain=prompt|model

In [16]:
chain.invoke({"messages":[HumanMessage(content="Hi My name is Akshat")]})

AIMessage(content='Hello Akshat! Nice to meet you. How can I help you today?', additional_kwargs={'reasoning_content': '<Think>\n\n</Think>'}, response_metadata={'token_usage': {'completion_tokens': 66, 'prompt_tokens': 283, 'total_tokens': 349, 'completion_time': 0.141644, 'completion_tokens_details': None, 'prompt_time': 0.011951, 'prompt_tokens_details': None, 'queue_time': 0.091525, 'total_time': 0.153595}, 'model_name': 'groq/compound', 'system_fingerprint': None, 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ea5ae-8d72-7dd2-ac99-497e9129b9ae-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 283, 'output_tokens': 66, 'total_tokens': 349})

In [17]:
with_message_history=RunnableWithMessageHistory(chain,get_session_history)

d:\udemy\python\venv\Lib\site-packages\IPython\core\interactiveshell.py:3747: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [18]:
config = {"configurable": {"session_id": "chat3"}}
response=with_message_history.invoke(
    [HumanMessage(content="Hi My name is Akshat")],
    config=config
)

response

AIMessage(content='Hello Akshat! Nice to meet you. How can I help you today?', additional_kwargs={'reasoning_content': '<Think>\n\n</Think>'}, response_metadata={'token_usage': {'completion_tokens': 56, 'prompt_tokens': 283, 'total_tokens': 339, 'completion_time': 0.121157, 'completion_tokens_details': None, 'prompt_time': 0.010919, 'prompt_tokens_details': None, 'queue_time': 0.105397, 'total_time': 0.132075}, 'model_name': 'groq/compound', 'system_fingerprint': None, 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ea5ae-8eb2-7133-8c4b-d92c61ae7393-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 283, 'output_tokens': 56, 'total_tokens': 339})

In [19]:
response = with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config,
)

response.content

'Your name is Akshat.'

In [20]:
## Add more complexity

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant. Answer all questions to the best of your ability in {language}.",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

chain = prompt | model

In [21]:
response=chain.invoke({"messages":[HumanMessage(content="Hi My name is Akshat")],"language":"Hindi"})
response.content

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01kd050gggeg8bh5xv33vpnr8x` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 5772, Requested 3154. Please try again in 6.945s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'compound', 'code': 'rate_limit_exceeded'}}

Let's now wrap this more complicated chain in a Message History class. This time, because there are multiple keys in the input, we need to specify the correct key to use to save the chat history.

In [ ]:
with_message_history=RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)

In [ ]:
config = {"configurable": {"session_id": "chat4"}}
repsonse=with_message_history.invoke(
    {'messages': [HumanMessage(content="Hi,I am Akshat. Can you give me some information about Python programming?")],"language":"Hindi"},
    config=config
)
repsonse.content

'**नमस्ते अक्षत!**  \nनीचे मैं आपको Python प्रोग्रामिंग के बारे में विस्तृत जानकारी दे रहा हूँ, जिसमें हमने पहले खोजे और संकलित किए हुए सभी मुख्य बिंदु शामिल हैं।\n\n---\n\n## 📌 Python क्या है?\nPython एक **उच्च‑स्तरीय, सामान्य‑उद्देश्य** प्रोग्रामिंग भाषा है। इसे 1991 में Guido van Rossum ने बनाया था और आज यह दुनिया की सबसे लोकप्रिय भाषाओं में से एक है। इसकी प्रमुख विशेषताएँ हैं:\n\n| विशेषता | विवरण |\n|--------|--------|\n| **सरल और पढ़ने‑योग्य सिंटैक्स** | कोड अंग्रेज़ी जैसी दिखता है, इंडेंटेशन (स्पेस/टैब) से ब्लॉक्स बनते हैं, जिससे शुरुआती भी जल्दी समझ सकें। |\n| **डायनामिक टाइपिंग** | वेरिएबल की डेटा टाइप को स्पष्ट रूप से घोषित करने की जरूरत नहीं; रन‑टाइम पर तय होती है। |\n| **ऑब्जेक्ट‑ओरिएंटेड** | क्लास, इनहेरिटेंस, एन्कैप्सुलेशन आदि को पूरी तरह सपोर्ट करता है। |\n| **मल्टी‑पैराडाइम** | प्रोसीजरल, ऑब्जेक्ट‑ओरिएंटेड, फ़ंक्शनल, इम्पेरेटिव आदि कई शैली में कोड लिखा जा सकता है। |\n| **बड़ी स्टैंडर्ड लाइब्रेरी** | “बैटरिज‑इनक्लूडेड” लाइब्रेरी के कारण फ़ाइल I/O, नेटवर्क, वेब, डेटाबेस, 

In [ ]:
response = with_message_history.invoke(
    {"messages": [HumanMessage(content="whats my name?")], "language": "Hindi"},
    config=config,
)

In [ ]:
response.content

'आपका नाम **अक्षत** है।  \n\n**कारण:** आपने अपने प्रारम्भिक संदेश में लिखा था, “Hi, I am Akshat.” इसलिए आपके द्वारा स्वयं प्रदान की गई जानकारी के आधार पर आपका नाम अक्षत ही है।'

### Managing the Conversation History
One important concept to understand when building chatbots is how to manage conversation history. If left unmanaged, the list of messages will grow unbounded and potentially overflow the context window of the LLM. Therefore, it is important to add a step that limits the size of the messages you are passing in.
'trim_messages' helper to reduce how many messages we're sending to the model. The trimmer allows us to specify how many tokens we want to keep, along with other parameters like if we want to always keep the system message and whether to allow partial messages

In [22]:
!pip install transformers

In [32]:
from langchain_core.messages import SystemMessage,trim_messages
trimmer=trim_messages(
    max_tokens=45,
    strategy="last",
    # token_counter="approximate",
    token_counter=model,
    include_system=True,
    allow_partial=False,
    start_on="human"
)
messages = [
    SystemMessage(content="you're a good assistant"),
    HumanMessage(content="hi! I'm bob"),
    AIMessage(content="hi!"),
    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="whats 2 + 2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!"),
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!"),
]
trimmer.invoke(messages)

[SystemMessage(content="you're a good assistant", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='I like vanilla ice cream', additional_kwargs={}, response_metadata={}),
 AIMessage(content='nice', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='whats 2 + 2', additional_kwargs={}, response_metadata={}),
 AIMessage(content='4', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='no problem!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='having fun?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='yes!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [33]:
from operator import itemgetter

from langchain_core.runnables import RunnablePassthrough

chain=(
    RunnablePassthrough.assign(messages=itemgetter("messages")|trimmer)
    | prompt
    | model
    
)

response=chain.invoke(
    {
    "messages":messages + [HumanMessage(content="What ice cream do i like")],
    "language":"English"
    }
)
response.content

'I don’t have any personal information about your taste, so I can’t pinpoint the exact flavor you prefer.\u202fHowever, based on the research I gathered earlier, the most popular (and therefore most‑commonly‑liked) ice‑cream flavors in the United States are:\n\n1. **Vanilla** – the overall #1 favorite, loved for its simplicity and versatility.  \n2. **Chocolate** – a close second, rich and indulgent.  \n3. **Strawberry** – often tops regional charts and is a classic fruit flavor.  \n4. **Cookies\u202f&\u202fCream** – a modern classic that mixes vanilla ice cream with cookie pieces.  \n5. **Mint Chocolate Chip** – a refreshing mint base with chocolate chunks.\n\nIf any of those sound like something you enjoy, there’s a good chance one of them is your go‑to flavor. If you’d like a more precise answer, feel free to give me a hint (e.g., “I love fruity desserts” or “I’m a fan of chocolate”) and I can narrow it down further!'

In [34]:
response = chain.invoke(
    {
        "messages": messages + [HumanMessage(content="what math problem did i ask")],
        "language": "English",
    }
)
response.content

'The math problem you originally asked was:\n\n**“What’s 2\u202f+\u202f2?”**\n\n**Reasoning and context**\n\n- In our earlier exchange you typed “whats 2 + 2”.\n- I responded with the calculation, confirming that 2\u202f+\u202f2 equals 4.\n- I even ran a quick Python check (`print(2 + 2)`) which output `4`, verifying the answer.\n\nSo the specific math problem you asked about was the addition of the numbers 2 and 2.'

In [35]:
## Lets wrap this in the MEssage History
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages",
)
config={"configurable":{"session_id":"chat5"}}

d:\udemy\python\venv\Lib\site-packages\IPython\core\interactiveshell.py:3747: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [36]:
response = with_message_history.invoke(
    {
        "messages": messages + [HumanMessage(content="what's my name?")],
        "language": "English",
    },
    config=config,
)

response.content

'I’m sorry, but I don’t have any information about your name. In our earlier exchange I noted that there’s no context or prior data that would let me infer or retrieve your name, and I have no way to look it up. Therefore, I’m unable to provide an answer to “what’s my name?” based on the information we’ve exchanged so far.'

In [37]:
response = with_message_history.invoke(
    {
        "messages": [HumanMessage(content="what math problem did i ask?")],
        "language": "English",
    },
    config=config,
)

response.content

'You haven’t actually asked a math problem yet. The only question you’ve posted so far is “what math problem did I ask?”—which is a meta‑question about a problem that wasn’t supplied. If you have a specific math problem you’d like help with, just let me know and I’ll be happy to work through it with you!'